In [ ]:
# Public-release setup: run from any working directory.
from pathlib import Path
import sys

def find_release_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "docs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the public release directory.")

PROJECT_ROOT = find_release_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PROJECT_ROOT / "results"  # User-supplied artifacts; not included in the release.
FIGURES_ROOT = PROJECT_ROOT / "figures"


# Narrow-Cone HH Alignment Figures

This notebook produces the paper figures for the narrow-cone effect in pair-wise hidden-state alignments `<h_i, h_j>`.

It reuses the `pair_metrics.csv` format and loading convention from `analyze_hh_alignment.ipynb`: each row is one hidden-state pair at one stored layer, with `dot`, `cosine`, and `metric_variant` columns. The default paper statistic below uses `metric_variant == "raw"`.

Layer indices follow the stored HH pipeline convention. In the current repository outputs, the stored `layer` values are the actual layer indices recorded in `pair_metrics.csv`; depending on how the HH extraction was launched, these may be a sparse subset of Transformer layers rather than every block. The special layer `-1` is resolved to the last available stored layer for each model.

## 1. Imports and Plotting Configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(context="paper", style="whitegrid", font_scale=1.0)
except Exception:
    sns = None
    plt.style.use("default")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

# Resolve paths relative to the repository root, not the notebook directory.
# This notebook lives under notebook_for_paper/, so Path("./results") can point to
# notebook_for_paper/results if the Jupyter kernel starts there.
PROJECT_ROOT = find_release_root()
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "figures/hh_narrow_cone"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT.resolve()}")
print(f"RESULTS_DIR = {RESULTS_DIR.resolve()}")
print(f"FIGURE_DIR = {FIGURE_DIR.resolve()}")

CSV_GLOB = "**/pair_metrics.csv"
METRIC_VARIANT = "raw"

HISTOGRAM_LAYERS = [0, 4, 8, 12, 20, -1]
LAYER_COLORS = {
    0: "#4C78A8",
    4: "#F58518",
    8: "#54A24B",
    12: "#E45756",
    20: "#72B7B2",
    -1: "#B279A2",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## 2. Model Configuration

The rows and panels use the exact model order requested for the paper. `model_tag` is the directory name used by the existing HH output pipeline.

In [ ]:
MODELS = [
    {"hf_name": "Qwen/Qwen2.5-0.5B", "model_tag": "Qwen2.5-0.5B", "short_name": "Qwen2.5-0.5B"},
    {"hf_name": "Qwen/Qwen2.5-1.5B", "model_tag": "Qwen2.5-1.5B", "short_name": "Qwen2.5-1.5B"},
    #{"hf_name": "Qwen/Qwen2.5-3B", "model_tag": "Qwen2.5-3B", "short_name": "Qwen2.5-3B"},
    {"hf_name": "Qwen/Qwen2.5-7B-Instruct", "model_tag": "Qwen2.5-7B-Instruct", "short_name": "Qwen2.5-7B-Inst."},
    {"hf_name": "meta-llama/Llama-3.2-3B", "model_tag": "Llama-3.2-3B", "short_name": "Llama3.2-3B"},
    {"hf_name": "mistralai/Mistral-7B-v0.1", "model_tag": "Mistral-7B-v0.1", "short_name": "Mistral-7B"},
    {"hf_name": "allenai/OLMoE-1B-7B-0125", "model_tag": "OLMoE-1B-7B-0125", "short_name": "OLMoE-1B-7B"},
]

# Use only the all-layer extended HH run for paper figures.
# This avoids silently falling back to older selected-layer outputs, which would make
# the heatmaps incomplete. Histograms still use HISTOGRAM_LAYERS below.
PREFERRED_EXPERIMENTS = [
    "hh_dataset_combos_extended_all_layers/gsm8k_to_mmlu",
]

## 3. Data Loading

These functions mirror the `analyze_hh_alignment.ipynb` convention: locate `pair_metrics.csv`, infer light metadata from the path, and filter to the selected `metric_variant`.

In [ ]:
REQUIRED_COLUMNS = {"layer", "pair_index", "left_index", "right_index", "dot", "cosine", "metric_variant"}
NUMERIC_COLUMNS = ["layer", "pair_index", "left_index", "right_index", "dot", "cosine"]


def infer_metadata_from_path(path, results_dir=RESULTS_DIR):
    path = Path(path)
    try:
        parts = path.relative_to(results_dir).parts
    except ValueError:
        parts = path.parts
    return {
        "source_csv": str(path),
        "experiment": parts[0] if len(parts) > 2 else None,
        "model": parts[1] if len(parts) > 3 else None,
    }


def load_metric_csv(path, results_dir=RESULTS_DIR):
    df = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    for col in NUMERIC_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    meta = infer_metadata_from_path(path, results_dir=results_dir)
    for key, value in meta.items():
        df[key] = value
    return df


def candidate_paths_for_model(model_tag, results_dir=RESULTS_DIR, experiments=PREFERRED_EXPERIMENTS):
    return [results_dir / exp / model_tag / "hh_alignment" / "pair_metrics.csv" for exp in experiments]


def resolve_model_csv(model_tag, results_dir=RESULTS_DIR, experiments=PREFERRED_EXPERIMENTS):
    """Resolve only approved paper result roots; do not fall back to older selected-layer runs."""
    for path in candidate_paths_for_model(model_tag, results_dir, experiments):
        if path.is_file():
            return path
    return None


def load_model_metrics(model_cfg, metric_variant=METRIC_VARIANT):
    path = resolve_model_csv(model_cfg["model_tag"])
    if path is None:
        warnings.warn(f"Missing HH pair_metrics.csv for {model_cfg['hf_name']}")
        return None, None
    df = load_metric_csv(path)
    df = df[df["metric_variant"] == metric_variant].copy()
    if df.empty:
        warnings.warn(f"No rows for metric_variant={metric_variant!r} in {path}")
    df["model_tag"] = model_cfg["model_tag"]
    df["short_name"] = model_cfg["short_name"]
    df["hf_name"] = model_cfg["hf_name"]
    return df, path


model_data = {}
load_report = []
for cfg in MODELS:
    df, path = load_model_metrics(cfg)
    model_data[cfg["model_tag"]] = df
    if df is None:
        load_report.append({"model": cfg["short_name"], "status": "missing", "source_csv": None, "rows": 0, "layers": []})
    else:
        layers = sorted(int(x) for x in df["layer"].dropna().unique())
        load_report.append({"model": cfg["short_name"], "status": "loaded", "source_csv": str(path), "rows": len(df), "n_layers": len(layers), "final_layer": layers[-1] if layers else None, "all_layers_available": layers == list(range(layers[-1] + 1)) if layers else False, "layers": layers})

load_report_df = pd.DataFrame(load_report)
load_report_df

## 4. Helper Functions

In [ ]:
def available_layers(df):
    if df is None or df.empty:
        return []
    return sorted(int(x) for x in df["layer"].dropna().unique())


def resolve_layer(df, requested_layer):
    layers = available_layers(df)
    if not layers:
        return None
    if requested_layer == -1:
        return layers[-1]
    return int(requested_layer) if int(requested_layer) in layers else None


def layer_label(requested_layer, resolved_layer):
    if requested_layer == -1:
        return f"final ({resolved_layer})"
    return f"layer {resolved_layer}"


def values_for_layer(df, metric, layer):
    if df is None or df.empty or layer is None:
        return np.array([])
    vals = df.loc[df["layer"] == layer, metric].dropna().to_numpy(dtype=float)
    return vals[np.isfinite(vals)]


def positive_ratio_by_layer(df, metric):
    if df is None or df.empty:
        return pd.DataFrame(columns=["layer", "positive_ratio", "n"])
    out = (df.groupby("layer", dropna=True)[metric]
           .agg(positive_ratio=lambda x: float((x.dropna() >= 0).mean()), n="count")
           .reset_index()
           .sort_values("layer"))
    out["layer"] = out["layer"].astype(int)
    return out


def save_figure(fig, stem):
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    png_path = FIGURE_DIR / f"{stem}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print(f"saved {pdf_path}")
    print(f"saved {png_path}")


def robust_xlim(values, lower=0.01, upper=0.99, pad_fraction=0.08):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return (-1.0, 1.0)
    lo, hi = np.quantile(values, [lower, upper])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(np.nanmin(values)), float(np.nanmax(values))
    if lo == hi:
        delta = max(abs(lo) * 0.1, 1.0)
        return lo - delta, hi + delta
    span = hi - lo
    lo -= pad_fraction * span
    hi += pad_fraction * span
    if lo < 0 < hi:
        limit = max(abs(lo), abs(hi))
        return -limit, limit
    return lo, hi

## 5. Distribution Figures

In [ ]:
def plot_histogram_grid(metric, stem, xlabel, common_xlim=None, bins=60):
    fig, axes = plt.subplots(2, 3, figsize=(11.2, 4.0), sharey=False)
    axes = axes.ravel()
    legend_handles = []
    legend_labels = []

    for ax, cfg in zip(axes, MODELS):
        df = model_data.get(cfg["model_tag"])
        ax.axvline(0, color="black", linewidth=1.0, alpha=0.8, zorder=3)
        ax.set_title(cfg["short_name"])
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Count")

        if df is None or df.empty:
            ax.text(0.5, 0.5, "missing data", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        panel_values = []
        resolved_seen = set()
        for req in HISTOGRAM_LAYERS:
            resolved = resolve_layer(df, req)
            if resolved is None:
                warnings.warn(f"{cfg['short_name']}: requested layer {req} is unavailable; skipping")
                continue
            if resolved in resolved_seen:
                continue
            resolved_seen.add(resolved)
            vals = values_for_layer(df, metric, resolved)
            if vals.size == 0:
                continue
            panel_values.append(vals)
            color = LAYER_COLORS[req]
            label = layer_label(req, resolved)
            ax.hist(vals, bins=bins, range=common_xlim, density=False, alpha=0.3, color=color, label=label, edgecolor="none")
            if label not in legend_labels:
                legend_handles.append(Line2D([0], [0], color=color, lw=6, alpha=0.45))
                legend_labels.append(label)

        if common_xlim is not None:
            ax.set_xlim(*common_xlim)
        elif panel_values:
            ax.set_xlim(*robust_xlim(np.concatenate(panel_values)))
        ax.grid(True, axis="y", linewidth=0.4, alpha=0.35)
        ax.grid(False, axis="x")

    fig.legend(legend_handles, legend_labels, loc="upper center", bbox_to_anchor=(0.5, 1.035), ncol=min(len(legend_labels), 6), frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_figure(fig, stem)
    return fig, axes


fig_cos_hist, axes_cos_hist = plot_histogram_grid(
    metric="cosine",
    stem="hh_cosine_layer_distributions",
    xlabel="Cosine similarity",
    common_xlim=(-1, 1),
    bins=np.linspace(-1, 1, 61),
)
plt.show()

fig_dot_hist, axes_dot_hist = plot_histogram_grid(
    metric="dot",
    stem="hh_dot_layer_distributions",
    xlabel="Dot product",
    common_xlim=None,
    bins=60,
)
plt.show()

## 6. Positive-Ratio Computation

In [ ]:
def build_positive_ratio_table(metric):
    rows = []
    for cfg in MODELS:
        df = model_data.get(cfg["model_tag"])
        pr = positive_ratio_by_layer(df, metric)
        if pr.empty:
            rows.append({"model": cfg["short_name"], "model_tag": cfg["model_tag"], "layer": np.nan, "positive_ratio": np.nan, "n": 0, "metric": metric})
            continue
        for _, row in pr.iterrows():
            rows.append({"model": cfg["short_name"], "model_tag": cfg["model_tag"], "layer": int(row["layer"]), "positive_ratio": float(row["positive_ratio"]), "n": int(row["n"]), "metric": metric})
    out = pd.DataFrame(rows)
    valid = out["positive_ratio"].dropna()
    if not ((valid >= 0).all() and (valid <= 1).all()):
        raise AssertionError(f"Found {metric} positive ratios outside [0, 1]")
    return out


cosine_positive = build_positive_ratio_table("cosine")
dot_positive = build_positive_ratio_table("dot")

cosine_positive.head(), dot_positive.head()

## 7. Heatmaps

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


def plot_positive_ratio_heatmap(ratio_df, stem, title):
    mat, layer_columns = positive_ratio_matrix(ratio_df)
    masked = np.ma.masked_invalid(mat)

    # Muted diverging palette, close to the style used in the Chapter 2 heatmap:
    # low -> muted green/gray, middle -> warm off-white, high -> muted red
    cmap = LinearSegmentedColormap.from_list(
        "narrow_cone_diverging",
        [
            "#c97b74",   # low: muted red
            "#e7c4bf",
            "#f2f4f6",   # center
            "#8fb3d9",
            "#1f5a99",   # high: blue
        ],
    ).copy()

    # Missing layers
    cmap.set_bad("#eef3f8")

    # 0.5 = neutral / chance-level sign balance
    norm = TwoSlopeNorm(
        vmin=0.0,
        vcenter=0.5,
        vmax=1.0,
    )

    n_layers = max(len(layer_columns), 1)

    # Make cells much closer to square.
    # 0.27 inch per layer gives a compact appendix figure.
    fig_width = max(8.0, 0.27 * n_layers + 2.5)
    fig_height = 2.8

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # pcolormesh gives explicit cells and subtle boundaries
    im = ax.pcolormesh(
        masked,
        cmap=cmap,
        norm=norm,
        shading="flat",
        edgecolors=(1, 1, 1, 0.55),
        linewidth=0.45,
    )
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if np.isnan(mat[i, j]):
                ax.plot(
                    [j + 0.18, j + 0.82],
                    [i + 0.82, i + 0.18],
                    color="#9aa3aa",
                    linewidth=0.7,
                    alpha=0.8,
                    solid_capstyle="round",
                )
    # Put row 0 at the top, like a standard heatmap
    ax.invert_yaxis()

    # -------------------------
    # Axes
    # -------------------------
    ax.set_title(title, fontsize=11, pad=6)
    ax.set_xlabel("Layer")
    ax.set_ylabel("")

    ax.set_yticks(np.arange(len(MODELS)) + 0.5)
    ax.set_yticklabels(
        [cfg["short_name"] for cfg in MODELS],
        fontsize=9,
    )

    if layer_columns:
        tick_step = max(1, int(np.ceil(len(layer_columns) / 12)))
        tick_idx = np.arange(0, len(layer_columns), tick_step)

        ax.set_xticks(tick_idx + 0.5)
        ax.set_xticklabels(
            [str(layer_columns[i]) for i in tick_idx],
            fontsize=8,
        )
    else:
        ax.set_xticks([])

    # Remove ordinary axis grid; cell borders already provide structure
    ax.grid(False)

    # Minimal frame
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(length=0)

    # -------------------------
    # Colorbar on the left,
    # similar to the Chapter 2 figure
    # -------------------------
    cbar = fig.colorbar(
        im,
        ax=ax,
        location="right",
        fraction=0.025,
        pad=0.02,
    )
    cbar.set_label(r"Fraction $\geq 0$", fontsize=10)
    cbar.set_ticks([0.0, 0.5, 0.8, 0.9, 1.0])
    cbar.ax.tick_params(labelsize=8, length=2)

    cbar.outline.set_visible(False)

    fig.tight_layout()

    save_figure(fig, stem)

    return fig, ax, mat, layer_columns

In [ ]:
fig_cos_heat, ax_cos_heat, cosine_matrix, cosine_layers = plot_positive_ratio_heatmap(
    cosine_positive,
    stem="hh_cosine_positive_ratio_heatmap",
    title="Cosine positive ratio",
)
plt.show()

fig_dot_heat, ax_dot_heat, dot_matrix, dot_layers = plot_positive_ratio_heatmap(
    dot_positive,
    stem="hh_dot_positive_ratio_heatmap",
    title="Dot-product positive ratio",
)
plt.show()

## 
8. Sanity Checks and Summary Statistics

In [ ]:
def final_layer_summary():
    rows = []
    for cfg in MODELS:
        df = model_data.get(cfg["model_tag"])
        layers = available_layers(df)
        if not layers:
            rows.append({"model": cfg["short_name"], "available_layers": 0, "stored_layers": [], "final_layer": np.nan, "final_cosine_positive_ratio": np.nan, "final_dot_positive_ratio": np.nan, "source_csv": None})
            continue
        final_layer = layers[-1]
        cos_vals = values_for_layer(df, "cosine", final_layer)
        dot_vals = values_for_layer(df, "dot", final_layer)
        source_csv = df["source_csv"].iloc[0] if "source_csv" in df.columns else None
        rows.append({"model": cfg["short_name"], "available_layers": len(layers), "stored_layers": layers, "final_layer": final_layer, "final_cosine_positive_ratio": float((cos_vals >= 0).mean()) if cos_vals.size else np.nan, "final_dot_positive_ratio": float((dot_vals >= 0).mean()) if dot_vals.size else np.nan, "source_csv": source_csv})
    return pd.DataFrame(rows)


summary_df = final_layer_summary()
display(summary_df)

for name, table in [("cosine", cosine_positive), ("dot", dot_positive)]:
    valid = table["positive_ratio"].dropna()
    print(f"\n{name} positive-ratio cells: {len(valid)}")
    for threshold in [0.5, 0.8, 0.9]:
        frac = float((valid >= threshold).mean()) if len(valid) else float("nan")
        print(f"  fraction >= {threshold:.1f}: {frac:.3f}")

missing_models = summary_df.loc[summary_df["available_layers"] == 0, "model"].tolist()
if missing_models:
    print("\nMissing HH data for:", ", ".join(missing_models))
    print("Run the HH alignment generation for these models before exporting final six-model paper figures.")